# **1. Import Lib**

In [4]:
# Cài đặt thư viện xử lý Transformer của HuggingFace
!pip install transformers torch

import pandas as pd
import numpy as np
import os
import pickle
import torch
import torch.nn as nn
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import BertTokenizer, BertModel
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

# Kết nối với Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


# **2. Prepare Data**

In [5]:
path_project = "/content/drive/MyDrive/FakeNewsDetection_Project"
path_dataset = os.path.join(path_project, "Dataset")

# Đọc dữ liệu từ Drive
df_true = pd.read_csv(os.path.join(path_dataset, "True.csv"))
df_fake = pd.read_csv(os.path.join(path_dataset, "Fake.csv"))

df_true['label'] = 1
df_fake['label'] = 0

df = pd.concat([df_true, df_fake], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df[['text', 'label']].dropna()

# Lấy tập mẫu nhỏ (ví dụ: 1500 dòng) để bảo vệ RAM của Colab
df_sub = df.sample(n=1500, random_state=42).reset_index(drop=True)
print(f"Dữ liệu sẵn sàng cho BERT + RNN: {len(df_sub)} mẫu.")

/tmp/ipykernel_1951/1468451772.py:6: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fake = pd.read_csv(os.path.join(path_dataset, "Fake.csv"))


Dữ liệu sẵn sàng cho BERT + RNN: 1500 mẫu.


# **3. BERT Tokenizer & DataLoader**

In [8]:
# Tải bộ mã hóa của BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
max_len = 128 # Giới hạn 128 từ đầu tiên của bài báo

# Ép kiểu dữ liệu cột text thành chuỗi văn bản để tránh lỗi định dạng dữ liệu trống
text_list = df_sub['text'].astype(str).tolist()

print("Đang mã hóa văn bản theo định dạng BERT... (Vui lòng đợi)")

# Cách viết chuẩn mới: Gọi trực tiếp tokenizer để xử lý danh bạ văn bản hàng loạt
encoded_batch = tokenizer(
    text_list,
    add_special_tokens=True,
    max_length=max_len,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt' # Trả về định dạng PyTorch Tensor trực tiếp
)

# Trích xuất dữ liệu Tensor từ kết quả mã hóa
input_ids = encoded_batch['input_ids']
attention_masks = encoded_batch['attention_mask']
labels = torch.tensor(df_sub['label'].values, dtype=torch.float32)

# Chia dữ liệu dạng Tensor theo tỷ lệ 80/20
train_inputs, test_inputs, train_labels, test_labels = train_test_split(
    input_ids, labels, test_size=0.2, random_state=42
)
train_masks, test_masks, _, _ = train_test_split(
    attention_masks, labels, test_size=0.2, random_state=42
)

# Tạo DataLoader để nạp dữ liệu theo từng Batch (giúp GPU không bị quá tải)
batch_size = 32

train_data = TensorDataset(train_inputs, train_masks, train_labels)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

test_data = TensorDataset(test_inputs, test_masks, test_labels)
test_sampler = SequentialSampler(test_data)
test_dataloader = DataLoader(test_data, sampler=test_sampler, batch_size=batch_size)

print("Đã sửa lỗi và chuẩn bị xong DataLoader cho mô hình!")

Đang mã hóa văn bản theo định dạng BERT... (Vui lòng đợi)
Đã sửa lỗi và chuẩn bị xong DataLoader cho mô hình!


# **4. BERT + RNN Model**

In [9]:
class BERT_RNN_Classifier(nn.Module):
    def __init__(self, hidden_dim=64):
        super(BERT_RNN_Classifier, self).__init__()
        # Nạp mô hình lõi BERT
        self.bert = BertModel.from_pretrained('bert-base-uncased')

        # Đóng băng BERT để không huấn luyện lại, chỉ lấy đặc trưng (Tiết kiệm thời gian)
        for param in self.bert.parameters():
            param.requires_grad = False

        # Lớp RNN nhận đầu vào 768 chiều từ BERT, đầu ra hidden_dim chiều
        self.rnn = nn.RNN(input_size=768, hidden_size=hidden_dim, batch_first=True)

        # Lớp Dropout chống quá khớp
        self.dropout = nn.Dropout(0.3)

        # Lớp tuyến tính đầu ra (Phân loại nhị phân)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        # 1. Đi qua BERT để lấy chuỗi đặc trưng ẩn
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # last_hidden_state có kích thước: [batch_size, max_len, 768]
        bert_embeddings = outputs.last_hidden_state

        # 2. Truyền chuỗi đặc trưng vào mạng RNN
        # rnn_out: [batch_size, max_len, hidden_dim]
        rnn_out, hidden = self.rnn(bert_embeddings)

        # Lấy trạng thái ẩn ở từ cuối cùng trong chuỗi để đại diện cho cả bài
        last_word_out = rnn_out[:, -1, :]

        # 3. Phân loại qua lớp Dense
        out = self.dropout(last_word_out)
        out = self.fc(out)
        return self.sigmoid(out)

# Khởi tạo mô hình và đẩy vào GPU T4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BERT_RNN_Classifier()
model.to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT_RNN_Classifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

# **5. Training Model**

In [11]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss() # Hàm mất mát cho phân loại nhị phân

epochs = 5
print("Đang huấn luyện tổ hợp BERT + RNN trên GPU T4...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_dataloader:
        b_input_ids, b_input_mask, b_labels = [t.to(device) for t in batch]

        model.zero_grad()
        outputs = model(b_input_ids, b_input_mask).squeeze()

        loss = criterion(outputs, b_labels)
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_dataloader):.4f}")

Đang huấn luyện tổ hợp BERT + RNN trên GPU T4...
Epoch 1/5 - Loss: 0.2487
Epoch 2/5 - Loss: 0.1760
Epoch 3/5 - Loss: 0.1420
Epoch 4/5 - Loss: 0.0993
Epoch 5/5 - Loss: 0.0742


# **6. Đánh giá kết quả & Tự sinh file model (APRF Tiêu chuẩn)**

In [12]:
model.eval()
predictions, true_labels = [], []

for batch in test_dataloader:
    b_input_ids, b_input_mask, b_labels = [t.to(device) for t in batch]
    with torch.no_grad():
        outputs = model(b_input_ids, b_input_mask).squeeze()

    # Chuyển xác suất sang nhãn 0/1
    preds = (outputs > 0.5).cpu().numpy().astype(int)
    predictions.extend(preds)
    true_labels.extend(b_labels.cpu().numpy().astype(int))

metrics = {
    'acc': accuracy_score(true_labels, predictions),
    'pre': precision_score(true_labels, predictions),
    'rec': recall_score(true_labels, predictions),
    'f1': f1_score(true_labels, predictions)
}

print("\n=== KẾT QUẢ THỰC NGHIỆM: BERT + RNN ===")
print(f"Accuracy:  {metrics['acc']:.4f}")
print(f"Precision: {metrics['pre']:.4f}")
print(f"Recall:    {metrics['rec']:.4f}")
print(f"F1-Score:  {metrics['f1']:.4f}")

# --- TỰ SINH FILE MODEL VÀ METADATA ---
path_models = os.path.join(path_project, "Models")
# Lưu trọng số mô hình dạng PyTorch (.pt)
torch.save(model.state_dict(), os.path.join(path_models, "bert_rnn_model.pt"))

# Ghi vào file Metadata chung
metadata_path = os.path.join(path_models, "model_info.txt")
with open(metadata_path, "a", encoding="utf-8") as f:
    f.write(f"- bert_rnn_model.pt: Accuracy {metrics['acc']:.4f}, dùng BERT-base + SimpleRNN.\n")

print("Đã sinh file model bert_rnn_model.pt và cập nhật ghi chú!")


=== KẾT QUẢ THỰC NGHIỆM: BERT + RNN ===
Accuracy:  0.9633
Precision: 0.9507
Recall:    0.9712
F1-Score:  0.9609
Đã sinh file model bert_rnn_model.pt và cập nhật ghi chú!
